# Day 2: Revolve the Profile into a 3D Model

Day 1 produced a metrically correct silhouette profile $r(z)$. This notebook
turns that profile into something you can actually hold in a viewer:

1. Re-extract the profile (same steps as Day 1, imported, not re-typed)
2. Fit the spline and integrate the volume analytically
3. Revolve it into a watertight mesh, export `cup.obj` / `cup.stl`
4. Sample an area-weighted point cloud, label it base / wall / lid, export `cup.ply`
5. Cross-check the mesh volume against the analytic integral

**Nothing here re-derives the scale.** The absolute size comes from the ruler
height in Day 1; if that number is wrong, everything below is wrong by the same
factor.

## 0. Setup

`cup_day1.py` is imported rather than copy-pasted. If I ever fix a bug in the
segmentation, this notebook picks it up instead of silently diverging.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np

BASE  = Path(r"C:\Users\SHUS\Documents\deepseek-harness\default-workspace\Rankross-drafts\Cup_Practice")
SHOTS = BASE / "shots2"
OUT   = BASE / "model"
OUT.mkdir(exist_ok=True)

sys.path.insert(0, str(BASE))
import cup_day1 as cd1          # reuse the exact Day-1 steps

print("output dir:", OUT)

## 1. Parameters

`HEIGHT_MM` is the only absolute number in this notebook. `ROI` crops the photo
so the desk edge cannot leak into the mask (that was one of the Day-1 bugs).

In [ ]:
HEIGHT_MM   = 213.0        # ruler measurement, to the top of the lid
ROI         = (700, 800, 400, 700)   # x, y, w, h around the cup
N_Z         = 300          # rings along the height
N_THETA     = 120          # samples around the axis
N_POINTS    = 50000        # point cloud size
BASE_MAX_Z  = 10.0         # mm: points below this are labelled "base"
LID_MIN_Z   = HEIGHT_MM - 32.0    # mm: points above this are labelled "lid"

LABEL_NAMES = {0: "base", 1: "wall", 2: "lid"}

## 2. Re-extract the profile

Same four steps as Day 1: mask from the background difference, per-row edges,
cut the contact shadow, convert to millimetres.

In [ ]:
img = cd1.imread_u(SHOTS / "cup.jpg")
bg  = cd1.imread_u(SHOTS / "bg.jpg")

x, y, w, h = ROI
img_c = img[y:y + h, x:x + w]
bg_c  = bg[y:y + h, x:x + w]

mask = cd1.build_mask(img_c, bg_c)
rows, left, right = cd1.profile_from_mask(mask)
rows, left, right = cd1.cut_base_contact(rows, left, right)

scale = HEIGHT_MM / float(rows.max() - rows.min() + 1)
z, r  = cd1.mask_to_profile_mm(rows, left, right, scale)
r     = cd1.fix_bad_rows(r)

print(f"mm/px      : {scale:.5f}")
print(f"rows       : {len(z)}")
print(f"z range    : {z.min():.2f} .. {z.max():.2f} mm")
print(f"max dia    : {2*np.percentile(r, 95):.2f} mm")
print(f"base dia   : {cd1.band_diameter(z, r, 0.05, 'bottom'):.2f} mm")

## 3. Spline fit and the analytic volume

The mesh built below is an approximation of this integral. If the two volumes
disagree, the mesh is wrong, not the profile.

In [ ]:
from scipy.interpolate import UnivariateSpline

spl, rms = cd1.fit_spline(z, r)

zz = np.linspace(z.min(), z.max(), N_Z)
rr = np.clip(spl(zz), 0.0, None)

V_analytic = float(np.pi * np.trapezoid(rr ** 2, zz) / 1000.0)   # mm^3 -> ml

print(f"spline rms     : {rms:.3f} mm")
print(f"V_analytic     : {V_analytic:.2f} ml")
print(f"r range        : {rr.min():.2f} .. {rr.max():.2f} mm")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 7))
ax.plot(r, z, ".", ms=2, color="#999", label="silhouette")
ax.plot(rr, zz, "-", lw=2, color="#1f77b4", label="spline")
ax.axhline(BASE_MAX_Z, color="#d62728", ls=":", lw=1)
ax.axhline(LID_MIN_Z, color="#2ca02c", ls=":", lw=1)
ax.set_xlabel("radius r (mm)")
ax.set_ylabel("height z (mm)")
ax.set_title(f"profile  |  V = {V_analytic:.1f} ml")
ax.grid(alpha=0.3)
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 4. Revolve into a mesh

A surface of revolution with $N_\theta$ samples per ring is

$$(r_i\cos\theta_j,\ r_i\sin\theta_j,\ z_i)$$

The sides are quads split into two triangles; the top and bottom are closed with
a triangle fan so the mesh is **watertight**, which is what makes the volume and
the STL valid.

In [ ]:
def revolve(zz, rr, n_theta=120):
    """Surface of revolution, closed at both ends. Returns (vertices, faces)."""
    th = np.linspace(0.0, 2.0 * np.pi, n_theta, endpoint=False)
    ct, st = np.cos(th), np.sin(th)

    V = np.empty((len(zz) * n_theta + 2, 3), dtype=np.float64)
    for i, (zi, ri) in enumerate(zip(zz, rr)):
        V[i * n_theta:(i + 1) * n_theta, 0] = ri * ct
        V[i * n_theta:(i + 1) * n_theta, 1] = ri * st
        V[i * n_theta:(i + 1) * n_theta, 2] = zi

    def idx(i, j):
        return i * n_theta + (j % n_theta)

    F = []
    for i in range(len(zz) - 1):
        for j in range(n_theta):
            a, b = idx(i, j), idx(i, j + 1)
            c, d = idx(i + 1, j + 1), idx(i + 1, j)
            F.append([a, b, c])
            F.append([a, c, d])

    # bottom cap
    ib = len(zz) * n_theta
    V[ib] = [0.0, 0.0, zz[0]]
    for j in range(n_theta):
        F.append([ib, idx(0, j + 1), idx(0, j)])

    # top cap
    it = ib + 1
    V[it] = [0.0, 0.0, zz[-1]]
    for j in range(n_theta):
        F.append([it, idx(len(zz) - 1, j), idx(len(zz) - 1, j + 1)])

    return V, np.asarray(F, dtype=np.int32)


def signed_volume(V, F):
    a, b, c = V[F[:, 0]], V[F[:, 1]], V[F[:, 2]]
    return float(np.einsum("ij,ij->i", a, np.cross(b, c)).sum() / 6.0)


Vc, Fc = revolve(zz, rr, N_THETA)

# guarantee outward-facing winding, otherwise the volume comes out negative
if signed_volume(Vc, Fc) < 0:
    Fc = Fc[:, ::-1]

print(f"vertices : {len(Vc)}")
print(f"triangles: {len(Fc)}")
print(f"volume   : {abs(signed_volume(Vc, Fc))/1000.0:.2f} ml")

## 5. Check the mesh, then export

`get_volume()` on a watertight mesh is exact for that mesh, so it is an
independent check on the hand-written integration above.

In [ ]:
import open3d as o3d

mesh = o3d.geometry.TriangleMesh(
    o3d.utility.Vector3dVector(Vc),
    o3d.utility.Vector3iVector(Fc),
)
mesh.compute_vertex_normals()

V_mesh = mesh.get_volume() / 1000.0     # mm^3 -> ml
err = 100.0 * (V_mesh - V_analytic) / V_analytic

print(f"watertight   : {mesh.is_watertight()}")
print(f"V mesh       : {V_mesh:.2f} ml")
print(f"V analytic   : {V_analytic:.2f} ml")
print(f"difference   : {err:+.2f} %")

In [ ]:
obj_path = OUT / "cup.obj"
stl_path = OUT / "cup.stl"

o3d.io.write_triangle_mesh(str(obj_path), mesh)
o3d.io.write_triangle_mesh(str(stl_path), mesh)

for p in (obj_path, stl_path):
    print(f"{p.name:10s} {p.stat().st_size/1024:8.1f} KB")

## 6. Sample a point cloud and label it

`sample_points_uniformly` is **area-weighted**, so the density is even on the
surface. Sampling $(\theta, z)$ on a grid instead would over-sample the narrow
ends, which is the mistake worth avoiding here.

In [ ]:
pcd = mesh.sample_points_uniformly(number_of_points=N_POINTS)
P = np.asarray(pcd.points)
N = np.asarray(pcd.normals)

labels = np.full(len(P), 1, dtype=np.int32)   # 1 = wall
labels[P[:, 2] <= BASE_MAX_Z] = 0             # 0 = base
labels[P[:, 2] >= LID_MIN_Z] = 2              # 2 = lid

print(f"points: {len(P)}")
for k, name in LABEL_NAMES.items():
    n = int((labels == k).sum())
    print(f"  {name:5s} {n:6d}  {100*n/len(P):5.1f}%")

In [ ]:
plt.figure(figsize=(4, 7))
cmap = np.array([[0.86, 0.24, 0.24], [0.78, 0.78, 0.78], [0.24, 0.47, 0.86]])
ax = plt.axes(projection="3d")
ax.scatter(P[:, 0], P[:, 1], P[:, 2], c=cmap[labels], s=0.4, linewidths=0)
ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)"); ax.set_zlabel("z (mm)")
ax.set_box_aspect((1, 1, 3.3))
ax.set_title("cup point cloud, labelled")
plt.tight_layout()
plt.show()

## 7. Save the point cloud

Two files on purpose:

- `cup.ply`, coloured by label, for looking at
- `cup_points.npz`, plain arrays, for the training loop

In [ ]:
ply_path = OUT / "cup.ply"
npz_path = OUT / "cup_points.npz"

pc = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(P))
pc.normals = o3d.utility.Vector3dVector(N)
pc.colors = o3d.utility.Vector3dVector(cmap[labels])
o3d.io.write_point_cloud(str(ply_path), pc)

np.savez_compressed(
    npz_path,
    points=P.astype(np.float32),
    normals=N.astype(np.float32),
    labels=labels,
    meta=json.dumps({"height_mm": HEIGHT_MM, "scale_mm_per_px": scale}),
)

for p in (ply_path, npz_path):
    print(f"{p.name:18s} {p.stat().st_size/1024:8.1f} KB")

## 8. Sanity checks that would catch a broken build

These are cheap and each one has a specific failure it catches.

In [ ]:
checks = []

lo, hi = P.min(axis=0), P.max(axis=0)
d_xy = 2.0 * float(np.abs(P[:, :2]).max())
checks.append(("height from points", hi[2] - lo[2], HEIGHT_MM, 1.0))
checks.append(("max diameter from points", d_xy, 65.0, 2.0))
checks.append(("mesh volume vs analytic", V_mesh, V_analytic, 1.0))

print(f"{'check':28s} {'value':>10s} {'ref':>10s} {'err%':>8s}  status")
print("-" * 66)
for name, val, ref, tol in checks:
    e = 100.0 * (val - ref) / ref
    print(f"{name:28s} {val:10.2f} {ref:10.2f} {e:+8.2f}  {'PASS' if abs(e) <= tol else 'FAIL'}")

## 9. Results

Fill this in after running, then copy it into `README.md`.

| Check | Value | Reference | Error |
| :--- | ---: | ---: | ---: |
| Mesh watertight | | required | |
| Mesh volume | | analytic | |
| Height from point cloud | | 213 mm | |
| Max diameter from point cloud | | 65 mm | |
| Label balance (base/wall/lid) | | 鈥?| |

**Open the model**: `model\cup.obj` in Windows 3D Viewer, or `model\cup.ply` in
Blender / CloudCompare.

**What is still missing**: this mesh is the *outer* surface only. There is no
wall thickness and no cavity, so it cannot be printed as a usable cup and its
volume is the exterior envelope, not the capacity.